# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Assem-ElQersh/FlyRank-ML-Internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

**Finding 1:** *"Growing content is 37.6% longer and 20% younger."* (Page 6)

**Methodology Question:** Does the validation design support the claim? Because the study splits by rising vs. falling impressions without controlling for age first, could it be that newer content naturally has rising impressions simply because it started at 0? Was content age controlled for before comparing word count, or is the length difference just an artifact of newer pages being longer?

**Finding 2:** *"AI Traffic: A Different Signal. The AI bucket behaves differently from classic organic winners."* (Page 11)

**Methodology Question:** Where does the label come from? How are "known AI referrals" defined and isolated in the GA4 data? Since attribution is famously difficult for AI bots and apps, does the population selection (only 'known' referrals) introduce survivor bias?

In [ ]:
# This cell is for CODE (numbers, a query, a check).
print("Methodology questions formulated and documented.")


## 2. My model under an honest split (before/after)

In Week 5, we correctly used a `GroupShuffleSplit`. Here is the before/after demonstration of what would have happened if we used a naive `train_test_split` (Random) compared to our honest Grouped Split. The Random Split allows pages from the same client into both train and test sets, enabling the model to "cheat" by memorizing client-specific baseline traffic rather than learning true content-decay signals.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit, train_test_split

url = 'https://raw.githubusercontent.com/Assem-ElQersh/FlyRank-ML-Internship/main/data/raw/content_refresh_anonymized.csv'
df = pd.read_csv(url)
df['is_declining'] = (df['trend_direction'] == 'down').astype(int)

features = ['days_since_last_update', 'impressions_90d', 'impressions_last_30d', 'word_count', 'avg_position', 'content_age_days']
df[features] = df[features].fillna(0)

def precision_at_k(scores, labels, k=50):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

# 1. Random Split (BEFORE - The Naive Approach)
train_df_rand, test_df_rand = train_test_split(df, test_size=0.3, random_state=42)

model_rand = RandomForestClassifier(max_depth=5, random_state=42)
model_rand.fit(train_df_rand[features], train_df_rand['is_declining'])
test_df_rand = test_df_rand.copy()
test_df_rand['pred_prob'] = model_rand.predict_proba(test_df_rand[features])[:, 1]
p50_rand = precision_at_k(test_df_rand['pred_prob'], test_df_rand['is_declining'])

# 2. Grouped Split (AFTER - The Honest Approach)
gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df['client_id']))
train_df_grp, test_df_grp = df.iloc[train_idx].copy(), df.iloc[test_idx].copy()

model_grp = RandomForestClassifier(max_depth=5, random_state=42)
model_grp.fit(train_df_grp[features], train_df_grp['is_declining'])
test_df_grp['pred_prob'] = model_grp.predict_proba(test_df_grp[features])[:, 1]
p50_grp = precision_at_k(test_df_grp['pred_prob'], test_df_grp['is_declining'])

print("--- SPLIT HONESTY COMPARISON ---")
print(f"Naive Random Split Precision@50:   {p50_rand:.1%}")
print(f"Honest Grouped Split Precision@50: {p50_grp:.1%}")
print("\nThe random split inflates the score because the model memorized client-specific baselines.")


## 3. Leakage audit

Our Week 5 model had a hidden leakage trap in the features list:
- The target `is_declining` is derived from `trend_direction`, which is computed from `trend_pct`.
- `trend_pct` mathematically relies on `impressions_last_30d` and `impressions_prev_30d`.
- In Week 5, we included `impressions_last_30d` and `impressions_90d` as features! This is massive target leakage (future/overlapping windows).

Below, we train a model WITH the leaky features, and one WITHOUT them, to show how much the leak artificially boosted performance.

In [ ]:
# Train WITH Leakage (Week 5 Baseline)
leaky_features = ['days_since_last_update', 'impressions_90d', 'impressions_last_30d', 'word_count', 'avg_position', 'content_age_days']
model_leaky = RandomForestClassifier(max_depth=5, random_state=42)
model_leaky.fit(train_df_grp[leaky_features], train_df_grp['is_declining'])
test_df_grp['prob_leaky'] = model_leaky.predict_proba(test_df_grp[leaky_features])[:, 1]
p50_leaky = precision_at_k(test_df_grp['prob_leaky'], test_df_grp['is_declining'])

# Train WITHOUT Leakage (Honest Features Only)
honest_features = ['days_since_last_update', 'word_count', 'avg_position', 'content_age_days']
model_honest = RandomForestClassifier(max_depth=5, random_state=42)
model_honest.fit(train_df_grp[honest_features], train_df_grp['is_declining'])
test_df_grp['prob_honest'] = model_honest.predict_proba(test_df_grp[honest_features])[:, 1]
p50_honest = precision_at_k(test_df_grp['prob_honest'], test_df_grp['is_declining'])

print("--- LEAKAGE AUDIT ---")
print(f"Precision@50 WITH Leakage:    {p50_leaky:.1%}")
print(f"Precision@50 WITHOUT Leakage: {p50_honest:.1%}")
print("\nThe leak gave us false confidence. The honest features yield a much lower, but realistic, score.")


## 4. Claim rewrite

**Original W5 Claim:** *"The model relies heavily on `impressions_90d` and `avg_position`. It avoids leakage because we did not give it any future-looking metrics like `trend_pct`."*

**Safe Rewrite:** *"In the observed sample, the model placed the highest descriptive importance on `impressions_90d` and `avg_position`. While we explicitly excluded `trend_pct`, our leakage audit revealed that overlapping time windows in the impression features introduced target information. These importances should be treated as directional decision-support rather than causal drivers."*

In [ ]:
print("Claim rewritten using safe, directional language.")


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.